In [1]:
import pyupbit
import pandas as pd
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta
import datetime as dt
import time
import logging

In [2]:
load_dotenv()

access_key = os.environ['access_key']
secret_key = os.environ['secret_key']

# 업비트 객체 생성
upbit = pyupbit.Upbit(access_key, secret_key)

In [3]:
# 보유 잔액 확인
# balances = upbit.get_balances(contain_req=True)
balances = upbit.get_balances()
print("보유 자산 목록:")
print(balances)

# 비트코인 현재 가격 조회
price = pyupbit.get_current_price("KRW-BTC")
print(f"현재 비트코인 가격: {price}원")


보유 자산 목록:
[{'currency': 'KRW', 'balance': '10000', 'locked': '0', 'avg_buy_price': '0', 'avg_buy_price_modified': True, 'unit_currency': 'KRW'}]
현재 비트코인 가격: 155560000.0원


In [4]:
chance = upbit.get_chance("KRW-BTC")
print(chance)

{'bid_fee': '0.0005', 'ask_fee': '0.0005', 'maker_bid_fee': '0.0005', 'maker_ask_fee': '0.0005', 'market': {'id': 'KRW-BTC', 'name': 'BTC/KRW', 'order_types': ['limit'], 'order_sides': ['ask', 'bid'], 'bid_types': ['best_fok', 'best_ioc', 'limit', 'limit_fok', 'limit_ioc', 'price'], 'ask_types': ['best_fok', 'best_ioc', 'limit', 'limit_fok', 'limit_ioc', 'market'], 'bid': {'currency': 'KRW', 'min_total': '5000'}, 'ask': {'currency': 'BTC', 'min_total': '5000'}, 'max_total': '1000000000', 'state': 'active'}, 'bid_account': {'currency': 'KRW', 'balance': '10000', 'locked': '0', 'avg_buy_price': '0', 'avg_buy_price_modified': True, 'unit_currency': 'KRW'}, 'ask_account': {'currency': 'BTC', 'balance': '0', 'locked': '0', 'avg_buy_price': '0', 'avg_buy_price_modified': False, 'unit_currency': 'KRW'}}


In [5]:
TICKER = "KRW-BTC"
def fetch_latest_candle(ticker: str = TICKER, interval: str = "minute240") -> pd.DataFrame | None:
    """
    Upbit API를 호출하여 가장 최신의 완성된 4시간봉 캔들 1개를 가져옵니다.
    'Open', 'High', 'Low', 'Close', 'Volume' 5개의 열로 구성된 DataFrame을 반환합니다.
    """
    logging.info(f"Fetching latest candle for {ticker} with interval {interval}...")
    try:
        # pyupbit.get_ohlcv는 가장 최신 봉부터 가져옵니다. count=1은 가장 최근에 완성된 봉 1개를 의미합니다.
        df = pyupbit.get_ohlcv(ticker, interval=interval, count=1)
        
        if df is None or df.empty:
            logging.error("Failed to fetch candle data from Upbit.")
            return None
        
        # 컬럼명을 대문자로 통일합니다.
        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Value']
        df.drop(columns=['Value'], inplace=True)
        df.index = df.index.tz_localize('Asia/Seoul').tz_convert('UTC')
        logging.info(f"New candle data fetched: \n{df}")
        return df

    except Exception as e:
        logging.error(f"An error occurred while fetching candle data: {e}")
        return None

fetch_latest_candle()

,Open,High,Low,Close,Volume
2025-08-26 04:00:00+00:00,155270000.0,155650000.0,155202000.0,155554000.0,134.814409


In [6]:
from pathlib import Path

DATA_DIR = Path("data")
CANDLE_DATA_PATH = DATA_DIR / "recent_candles.csv"
def load_recent_candles() -> pd.DataFrame:
    """저장된 최근 캔들 데이터를 불러옵니다."""
    if not CANDLE_DATA_PATH.exists():
        logging.warning("Candle data file not found. Returning empty DataFrame.")
        return pd.DataFrame()
    try:
        df = pd.read_csv(CANDLE_DATA_PATH, encoding="utf-8-sig", index_col=0, parse_dates=True)
        # df.index = pd.to_datetime(df['timestamp'], utc=True)
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
        logging.info(f"Loaded {len(df)} recent candles.")
        return df
    except Exception as e:
        logging.error(f"Error loading candle data: {e}")
        return pd.DataFrame()
load_recent_candles()

,Open,High,Low,Close,Volume
Datetime,,,,,
2025-06-20 12:00:00+00:00,146070000.0,146361000.0,144200000.0,144816000.0,224.001959
2025-06-20 16:00:00+00:00,144816000.0,144957000.0,143107000.0,144172000.0,270.529371
2025-06-20 20:00:00+00:00,144150000.0,144692000.0,143645000.0,143956000.0,162.209833
2025-06-21 00:00:00+00:00,143956000.0,144272000.0,143700000.0,144155000.0,132.443507
2025-06-21 04:00:00+00:00,144155000.0,144189000.0,143748000.0,143855000.0,83.486670
...,...,...,...,...,...
2025-08-25 08:00:00+00:00,155951000.0,156524000.0,155510000.0,156000000.0,480.354708
2025-08-25 12:00:00+00:00,156058000.0,157500000.0,155346000.0,156951000.0,440.391547
2025-08-25 16:00:00+00:00,156951000.0,158000000.0,156255000.0,156414000.0,162.358313


# 최근 Ticker 400개 가져오기

In [3]:
from datetime import timezone

now = datetime.now()
# now = pd.to_datetime("2025-08-26 01:00:00", format='%Y-%m-%d %H:%M:%S').tz_localize('Asia/Seoul')
now = now.astimezone(timezone.utc)
df1 = pyupbit.get_ohlcv("KRW-BTC", interval="minute240", to=now)
df1.index = df1.index.tz_localize('Asia/Seoul').tz_convert('UTC')
display(df1)

now = df1.iloc[0].name
df2 = pyupbit.get_ohlcv("KRW-BTC", interval="minute240", to=now)
df2.index = df2.index.tz_localize('Asia/Seoul').tz_convert('UTC')
display(df2)

df = pd.concat([df2, df1]).drop_duplicates()
df = df.sort_index()

df.rename(columns={"open":"Open", "high":"High", "low":"Low", "close":"Close", "volume":"Volume"}, inplace=True)
df = df[["Open", "High", "Low", "Close", "Volume"]]

df.index.name = "Datetime"
df.to_csv("./data/recent_candles.csv", encoding='utf-8-sig', index=True)
df

,open,high,low,close,volume,value
2025-07-24 12:00:00+00:00,161198000.0,162098000.0,160358000.0,161674000.0,335.503923,5.412430e+10
2025-07-24 16:00:00+00:00,161693000.0,161870000.0,160909000.0,161407000.0,101.144166,1.634089e+10
2025-07-24 20:00:00+00:00,161407000.0,161605000.0,160874000.0,160928000.0,137.002323,2.208001e+10
2025-07-25 00:00:00+00:00,160938000.0,161239000.0,158099000.0,158436000.0,886.257916,1.412659e+11
2025-07-25 04:00:00+00:00,158455000.0,159249000.0,157746000.0,158499000.0,713.428687,1.129547e+11
...,...,...,...,...,...,...
2025-08-26 00:00:00+00:00,155887000.0,155940000.0,153952000.0,155270000.0,830.946060,1.287269e+11
2025-08-26 04:00:00+00:00,155270000.0,155700000.0,155000000.0,155549000.0,268.314575,4.170406e+10
2025-08-26 08:00:00+00:00,155550000.0,155887000.0,154969000.0,155140000.0,213.455016,3.317337e+10
2025-08-26 12:00:00+00:00,155140000.0,156000000.0,154381000.0,154461000.0,446.322891,6.927456e+10


,open,high,low,close,volume,value
2025-06-21 04:00:00+00:00,144155000.0,144189000.0,143748000.0,143855000.0,83.486670,1.201873e+10
2025-06-21 08:00:00+00:00,143855000.0,144427000.0,143855000.0,144314000.0,75.415841,1.087787e+10
2025-06-21 12:00:00+00:00,144314000.0,144400000.0,144000000.0,144141000.0,67.813853,9.778206e+09
2025-06-21 16:00:00+00:00,144240000.0,144296000.0,143032000.0,143033000.0,105.493835,1.514332e+10
2025-06-21 20:00:00+00:00,143033000.0,143837000.0,141151000.0,142782000.0,627.924842,8.936154e+10
...,...,...,...,...,...,...
2025-07-23 16:00:00+00:00,161497000.0,161500000.0,160206000.0,161495000.0,133.424923,2.147399e+10
2025-07-23 20:00:00+00:00,161495000.0,162386000.0,159967000.0,161990000.0,472.505772,7.624787e+10
2025-07-24 00:00:00+00:00,162000000.0,162888000.0,161444000.0,161667000.0,370.274077,6.012675e+10
2025-07-24 04:00:00+00:00,161667000.0,161795000.0,159000000.0,160694000.0,895.168649,1.432170e+11


,Open,High,Low,Close,Volume
Datetime,,,,,
2025-06-21 04:00:00+00:00,144155000.0,144189000.0,143748000.0,143855000.0,83.486670
2025-06-21 08:00:00+00:00,143855000.0,144427000.0,143855000.0,144314000.0,75.415841
2025-06-21 12:00:00+00:00,144314000.0,144400000.0,144000000.0,144141000.0,67.813853
2025-06-21 16:00:00+00:00,144240000.0,144296000.0,143032000.0,143033000.0,105.493835
2025-06-21 20:00:00+00:00,143033000.0,143837000.0,141151000.0,142782000.0,627.924842
...,...,...,...,...,...
2025-08-26 00:00:00+00:00,155887000.0,155940000.0,153952000.0,155270000.0,830.946060
2025-08-26 04:00:00+00:00,155270000.0,155700000.0,155000000.0,155549000.0,268.314575
2025-08-26 08:00:00+00:00,155550000.0,155887000.0,154969000.0,155140000.0,213.455016


In [ ]:
logger = logging.getLogger()
def fetch_historical_candles_simple(ticker: str = "KRW-BTC", interval: str = "minute240") -> pd.DataFrame | None:
    """
    Upbit API를 두 번 호출하여 총 400개의 4시간봉 캔들을 가져옵니다.
    """
    logger.info("Fetching 400 historical candles in two chunks...")
    try:
        # 1. 가장 최신 캔들 200개를 가져옵니다.
        df1 = pyupbit.get_ohlcv(ticker, interval=interval, count=200)
        if df1 is None or df1.empty:
            logger.error("Failed to fetch candles.")
            return None
        time.sleep(0.2)
        oldest_timestamp = df1.index[0] - pd.Timedelta(hours=9)
        df2 = pyupbit.get_ohlcv(ticker, interval=interval, count=200, to=oldest_timestamp)
        if df2 is None or df2.empty:
            logger.error("Failed to fetch candles.")
            return None

        df_total = pd.concat([df2, df1])

        # 컬럼명 정리
        df_total.columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Value']
        df_total.drop(columns=['Value'], inplace=True)
        
        # 시간대(Timezone)를 UTC로 통일
        df_total.index = df_total.index.tz_localize('Asia/Seoul').tz_convert('UTC')
        
        # 시간순으로 정렬
        df_total.sort_index(inplace=True)
        
        logger.info(f"Successfully fetched a total of {len(df_total)} candles.")
        return df_total

    except Exception as e:
        logger.error(f"An error occurred while fetching historical candles: {e}")
        return None
df = fetch_historical_candles_simple()
df = df[~df.index.duplicated(keep='last')]
df

,Open,High,Low,Close,Volume
2025-06-21 04:00:00+00:00,144155000.0,144189000.0,143748000.0,143855000.0,83.486670
2025-06-21 08:00:00+00:00,143855000.0,144427000.0,143855000.0,144314000.0,75.415841
2025-06-21 12:00:00+00:00,144314000.0,144400000.0,144000000.0,144141000.0,67.813853
2025-06-21 16:00:00+00:00,144240000.0,144296000.0,143032000.0,143033000.0,105.493835
2025-06-21 20:00:00+00:00,143033000.0,143837000.0,141151000.0,142782000.0,627.924842
...,...,...,...,...,...
2025-08-26 00:00:00+00:00,155887000.0,155940000.0,153952000.0,155270000.0,830.946060
2025-08-26 04:00:00+00:00,155270000.0,155700000.0,155000000.0,155549000.0,268.314575
2025-08-26 08:00:00+00:00,155550000.0,155887000.0,154969000.0,155140000.0,213.455016
2025-08-26 12:00:00+00:00,155140000.0,156000000.0,154381000.0,154461000.0,446.322891


# 백트래킹

In [12]:
from datetime import timezone

def fetch_400_candles(now: datetime = datetime.now()):
    now = now.astimezone(timezone.utc)
    df1 = pyupbit.get_ohlcv("KRW-BTC", interval="minute240", to=now)
    df1.index = df1.index.tz_localize('Asia/Seoul').tz_convert('UTC')

    now = df1.iloc[0].name
    df2 = pyupbit.get_ohlcv("KRW-BTC", interval="minute240", to=now)
    df2.index = df2.index.tz_localize('Asia/Seoul').tz_convert('UTC')

    df = pd.concat([df2, df1]).drop_duplicates()
    df = df.sort_index()

    df.rename(columns={"open":"Open", "high":"High", "low":"Low", "close":"Close", "volume":"Volume"}, inplace=True)
    df = df[["Open", "High", "Low", "Close", "Volume"]]

    df.index.name = "Datetime"
    # df.to_csv("./data/recent_candles.csv", encoding='utf-8-sig', index=True)
    return df

now = pd.to_datetime("2025-08-01 09:00:00", format='%Y-%m-%d %H:%M:%S').tz_localize('Asia/Seoul')
df = fetch_400_candles(now)
df

,Open,High,Low,Close,Volume
Datetime,,,,,
2025-05-26 08:00:00+00:00,152517000.0,152699000.0,151997000.0,152277000.0,127.909150
2025-05-26 12:00:00+00:00,152277000.0,152950000.0,151951000.0,152658000.0,196.344528
2025-05-26 16:00:00+00:00,152658000.0,153000000.0,151509000.0,151766000.0,128.255840
2025-05-26 20:00:00+00:00,151766000.0,152685000.0,151613000.0,152195000.0,92.692234
2025-05-27 00:00:00+00:00,152197000.0,152350000.0,150001000.0,151265000.0,454.018050
...,...,...,...,...,...
2025-07-31 04:00:00+00:00,163500000.0,163600000.0,163057000.0,163419000.0,163.416061
2025-07-31 08:00:00+00:00,163420000.0,163598000.0,163251000.0,163378000.0,89.959903
2025-07-31 12:00:00+00:00,163321000.0,163997000.0,163000000.0,163996000.0,358.602329


In [23]:
TICKER = "KRW-BTC"
def fetch_latest_candle(ticker: str = TICKER, t = datetime.now(), interval: str = "minute240") -> pd.DataFrame | None:
    """
    Upbit API를 호출하여 가장 최신의 완성된 4시간봉 캔들 1개를 가져옵니다.
    'Open', 'High', 'Low', 'Close', 'Volume' 5개의 열로 구성된 DataFrame을 반환합니다.
    """
    try:
        # pyupbit.get_ohlcv는 가장 최신 봉부터 가져옵니다. count=1은 가장 최근에 완성된 봉 1개를 의미합니다.
        df = pyupbit.get_ohlcv(ticker, interval=interval, count=1, to=t)
        
        if df is None or df.empty:
            return None
        
        # 컬럼명을 대문자로 통일합니다.
        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Value']
        df.drop(columns=['Value'], inplace=True)
        df.index = df.index.tz_localize('Asia/Seoul').tz_convert('UTC')
        return df

    except Exception as e:
        # logger.error(f"An error occurred while fetching candle data: {e}")
        return None

In [ ]:
import logging
import joblib
import numpy as np
import pandas as pd
import pandas_ta as ta  # pandas_ta 라이브러리 사용
from keras.models import load_model

# --- 설정값 ---
LOOKBACK_WINDOW = 30
ENTRY_THRESHOLD = 0.2  # 최적화된 값으로 수정하세요
EXIT_THRESHOLD = 0.42   # 최적화된 값으로 수정하세요

 # 일봉 SMA(50) 계산에 필요한 최소 300개와 약간의 여유분을 고려합니다.
# LOOKBACK_WINDOW(30)보다 훨씬 긴 기간이 필요합니다.
MIN_CANDLE_COUNT = 400 # 50일 * 6개/일 = 300개 + 여유분

logger = logging.getLogger(__name__)

MODEL = load_model('./lstm/best_mtf_model_ver1.1.h5')
MODEL.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
SCALER = joblib.load('./lstm/best_mtf_scaler_ver1.1.pkl')
FIT_FEATURE_NAMES = SCALER.feature_names_in_


def generate_signal(candles_4h: pd.DataFrame) -> str:
    """
    주어진 캔들 데이터로 피처를 생성하고 모델 예측을 통해 신호를 반환합니다.
    """
    if MODEL is None or SCALER is None:
        return 'hold'

    if len(candles_4h) < MIN_CANDLE_COUNT:
        logger.warning(f"Not enough candle data ({len(candles_4h)}) to create features. Need at least {MIN_CANDLE_COUNT}.")
        return 'hold'

    logger.info("Creating features for prediction...")
    
    # 1. 피처 생성
    features_4h = candles_4h.copy()
    features_4h.ta.rsi(length=14, append=True, col_names=('RSI_14_4H',))
    features_4h.ta.macd(fast=12, slow=26, signal=9, append=True, col_names=('MACD_12_26_9_4H', 'MACDh_12_26_9_4H', 'MACDs_12_26_9_4H'))
    features_4h.ta.bbands(length=20, std=2, append=True, col_names=('BBL_20_2.0_4H', 'BBM_20_2.0_4H', 'BBU_20_2.0_4H', 'BBB_20_2.0_4H', 'BBP_20_2.0_4H'))

    daily_resampled = candles_4h.resample('D').agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
    })
    features_1d = daily_resampled.copy()
    features_1d.ta.rsi(length=14, append=True, col_names=('RSI_14_1D',))
    features_1d.ta.sma(length=50, append=True, col_names=('SMA_50_1D',))
    features_1d.ta.adx(length=14, append=True, col_names=('ADX_14_1D', 'DMP_14_1D', 'DMN_14_1D'))
    
    daily_indicator_cols = ['RSI_14_1D', 'SMA_50_1D', 'ADX_14_1D', 'DMP_14_1D', 'DMN_14_1D']
    features_1d_to_merge = features_1d[daily_indicator_cols]
    
    final_features = pd.merge(features_4h, features_1d_to_merge, left_index=True, right_index=True, how='left')
    final_features.ffill(inplace=True)
    final_features.dropna(inplace=True)

    if len(final_features) < LOOKBACK_WINDOW:
        logger.warning("Not enough data after feature creation to form a sequence.")
        return 'hold'

    # 2. 예측에 사용할 마지막 시퀀스 준비
    last_sequence_features = final_features[FIT_FEATURE_NAMES].tail(LOOKBACK_WINDOW)
    
    if len(last_sequence_features) < LOOKBACK_WINDOW:
        logger.warning("Sequence is shorter than lookback window. Skipping prediction.")
        return 'hold'

    scaled_features = SCALER.transform(last_sequence_features)
    X = np.array([scaled_features])

    # 3. 예측 수행
    logger.info("Predicting with the model...")
    prediction = MODEL.predict(X)[0]
    prob_loss, prob_hold, prob_profit = prediction[0], prediction[1], prediction[2]
    print(f"Prediction probabilities -> Loss: {prob_loss:.4f}, Hold: {prob_hold:.4f}, Profit: {prob_profit:.4f}")

    # 4. 신호 결정
    if prob_profit > ENTRY_THRESHOLD:
        return 'buy'
    elif prob_loss > EXIT_THRESHOLD:
        return 'sell'
    else:
        return 'hold'


In [ ]:
# UTC 기준 8월 1일 0시부터 실행
# from tqdm import tqdm
now = pd.to_datetime("2025-08-01 09:00:00", format='%Y-%m-%d %H:%M:%S').tz_localize('Asia/Seoul')
df = fetch_400_candles(now)

l = 156
for i in range(l):
    print(f"{i+1}/{l}")
    df_new = fetch_latest_candle(TICKER, df.index[-1]+timedelta(hours=4, minutes=1))
    df = pd.concat([df, df_new]).drop_duplicates()
    # print(df.index[0], df.index[-1])
    # print(generate_signal(df), df.index[-1])
df.index.name = "Datetime"
df.to_csv("./data/recent_candles_test.csv", encoding='utf-8-sig', index=True)

0/156
1/156
2/156
3/156
4/156
5/156
6/156
7/156
8/156
9/156
10/156
11/156
12/156
13/156
14/156
15/156
16/156
17/156
18/156
19/156
20/156
21/156
22/156
23/156
24/156
25/156
26/156
27/156
28/156
29/156
30/156
31/156
32/156
33/156
34/156
35/156
36/156
37/156
38/156
39/156
40/156
41/156
42/156
43/156
44/156
45/156
46/156
47/156
48/156
49/156
50/156
51/156
52/156
53/156
54/156
55/156
56/156
57/156
58/156
59/156
60/156
61/156
62/156
63/156
64/156
65/156
66/156
67/156
68/156
69/156
70/156
71/156
72/156
73/156
74/156
75/156
76/156
77/156
78/156
79/156
80/156
81/156
82/156
83/156
84/156
85/156
86/156
87/156
88/156
89/156
90/156
91/156
92/156
93/156
94/156
95/156
96/156
97/156
98/156
99/156
100/156
101/156
102/156
103/156
104/156
105/156
106/156
107/156
108/156
109/156
110/156
111/156
112/156
113/156
114/156
115/156
116/156
117/156
118/156
119/156
120/156
121/156
122/156
123/156
124/156
125/156
126/156
127/156
128/156
129/156
130/156
131/156
132/156
133/156
134/156
135/156
136/156
137/156
138/15